In [90]:
from pathlib import Path
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import softmax

In [91]:
ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
OOF_DIR = ROOT_DIR / "oof"

SUBMISSION_DIR = ROOT_DIR / "submission"
SUBMISSION_DIR.mkdir(exist_ok=True)

In [92]:
conv_test = np.load(OOF_DIR / "convnext_test_logits.npy")
swin_test = np.load(OOF_DIR / "swin_test_logits.npy")
sig_test  = np.load(OOF_DIR / "siglip_test_logits.npy")

In [93]:
p_conv = softmax(conv_test, axis=1)
p_swin = softmax(swin_test, axis=1)
p_sig = softmax(sig_test, axis=1)

In [94]:
weights = np.array([0.33, 0.33, 0.33])

eps = 1e-12

gamma = 1.10

p_conv = p_conv ** gamma
p_swin = p_swin ** gamma
p_sig = p_sig ** gamma

p_conv /= p_conv.sum(axis=1, keepdims=True)
p_swin /= p_swin.sum(axis=1, keepdims=True)
p_sig /= p_sig.sum(axis=1, keepdims=True)


log_prob = (
      weights[0] * np.log(p_conv + eps)
    + weights[1] * np.log(p_swin + eps)
    + weights[2] * np.log(p_sig + eps)
)

ensemble_prob = np.exp(log_prob)
ensemble_prob /= ensemble_prob.sum(axis=1, keepdims=True)

In [95]:
pred = ensemble_prob.argmax(axis=1)

In [96]:
confidence = ensemble_prob.max(axis=1)

margin = (
    np.sort(ensemble_prob, axis=1)[:, -1]
    -
    np.sort(ensemble_prob, axis=1)[:, -2]
)

In [97]:
detail = pd.DataFrame({

    "prediction": pred,

    "confidence": confidence,

    "margin": margin,

    "prob_recyclable": ensemble_prob[:,0],

    "prob_electronic": ensemble_prob[:,1],

    "prob_organic": ensemble_prob[:,2]

})

detail.to_csv(
    SUBMISSION_DIR/"test_predictions.csv",
    index=False
)

In [98]:
submission = pd.read_csv("submission.csv")

label_col = submission.columns[1]
submission[label_col] = pred.astype(np.int64)

print(submission.dtypes)

submission.to_csv(
    "submission_naudzubillah.csv",
    index=False
)

submission.head()

id           int64
predicted    int64
dtype: object


,id,predicted
0,1,2
1,2,2
2,3,2
3,4,1
4,5,0


In [99]:
print(submission.dtypes)

id           int64
predicted    int64
dtype: object


In [100]:
print(conv_test.shape)
print(swin_test.shape)
print(sig_test.shape)

(1458, 3)
(1458, 3)
(1458, 3)


In [101]:
print(len(submission))

1458
